In [4]:
"""
W&G Baird — Customer Value, Reorder & Churn Analysis Pipeline (Python port)
Author: Uzair Khan | QUB KTP recruitment task
"""

import re
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Config ---------------------------------------------------
INPUT_DIR = Path("data/input")     # drop any new export here, same column structure
OUTPUT_DIR = Path("output")
CHART_DIR = OUTPUT_DIR / "charts"  # PNG exports for the README / slide deck
CHURN_K = 1

MIN_ORDERS_FOR_REORDER_WINDOW = 3  # need >=3 orders on one Title to trust a cadence

# EUR -> GBP conversion
EUR_TO_GBP = 0.86

INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Known naming drift
KNOWN_FIELD_ALIASES = {"Rep Name": "Rep"}

RENAME_MAP = {
    "Title": "title",
    "CustomerID": "customer_id",
    "Customer Name": "customer_name",
    "Job Status": "job_status",
    "SalesIn": "sales_in",
    "SalesOut": "sales_out",
    "Ship date": "ship_date",
    "Quantity": "quantity",
    "Sell Price": "sell_price",
    "Mup%": "mup_pct",
    "VA Amount": "va_amount",
    "VA/24": "va_per_hour",
    "VA%": "va_pct",
    "VA/K": "va_per_k",
    "Rebate": "rebate",
    "Puchases": "purchases",
    "Press hrs": "press_hrs",
    "Impressions": "impressions",
    "Handling": "handling",
    "Labour": "labour",
    "Paper": "paper",
    "Rep": "rep",
    "Region": "region",
    "Industry": "industry",
    "Work Type": "work_type",
    "Product Type": "product_type",
    "Binding Type": "binding_type",
    "Currency": "currency",
}


def _snake(name: str) -> str:
    """Fallback snake_case for any column not covered by RENAME_MAP
    (e.g. 'Year', 'Week No.') — equivalent to janitor::clean_names()."""
    name = re.sub(r"[^0-9a-zA-Z]+", "_", str(name)).strip("_")
    name = re.sub(r"(?<!^)(?=[A-Z])", "_", name)
    return name.lower()


# 1. LOAD
def get_expected_columns(reference_file: Path) -> list:
    """Read the Field Definitions tab and return the expected column names.
    This is the source of truth for what a new data drop should contain."""
    defs = pd.read_excel(reference_file, sheet_name="Field Definitions")
    names_raw = defs["Field Name"].dropna().tolist()
    return [KNOWN_FIELD_ALIASES.get(n, n) for n in names_raw]


def validate_columns(file_path: Path, expected_cols: list) -> bool:

    actual_cols = pd.read_excel(file_path, sheet_name="Master Plain (Anon)", nrows=0).columns.tolist()

    missing = sorted(set(expected_cols) - set(actual_cols))
    unexpected = sorted(set(actual_cols) - set(expected_cols))

    if missing:
        print(f"MISSING columns in {file_path.name}: {', '.join(missing)}")
    if unexpected:
        print(f"UNEXPECTED columns in {file_path.name}: {', '.join(unexpected)}")

    return len(missing) == 0  # only missing columns block ingestion; extra columns just warn


def load_raw_data(input_dir: Path = INPUT_DIR) -> pd.DataFrame:

    files = sorted(input_dir.glob("*.xlsx"))
    if not files:
        raise FileNotFoundError(f"No .xlsx files found in {input_dir}")

    expected_cols = get_expected_columns(files[0])
    valid_files = [f for f in files if validate_columns(f, expected_cols)]
    if len(valid_files) < len(files):
        print(f"WARNING: {len(files) - len(valid_files)} file(s) skipped due to missing columns — see messages above.")

    frames = []
    for f in valid_files:
        d = pd.read_excel(f, sheet_name="Master Plain (Anon)")
        d["source_file"] = f.name
        frames.append(d)
    return pd.concat(frames, ignore_index=True)


# 1.5 QUICK PROFILING / QA CHECKS
def profile_data(raw: pd.DataFrame) -> None:
    """Independent re-run of the core profiling checks against the RAW data,
    before any cleaning is applied."""
    print("---- Null counts (top 10) ----")
    print(raw.isna().sum().sort_values(ascending=False).head(10))

    print("\n---- Fully identical duplicate rows ----")
    dup_mask = raw.duplicated(keep="first")
    dup_any = raw.duplicated(keep=False)
    print(f"{dup_mask.sum()} extra copies ( {dup_any.sum()} rows total involved )")

    print("\n---- Repeated Title values (legitimate reorders, not errors) ----")
    counts = raw["Title"].value_counts()
    repeated = counts[counts > 1]
    print(f"{repeated.sum()} rows across {len(repeated)} distinct Titles")

    print("\n---- #DIV/0! rows ----")
    print("VA%: ", (raw["VA%"] == "#DIV/0!").sum())
    print("Mup%:", (raw["Mup%"] == "#DIV/0!").sum())

    print("\n---- Ship date before SalesIn (impossible) ----")
    print(f"{(raw['Ship date'] < raw['SalesIn']).sum()} rows")


# 2. CLEAN
def clean_data(raw: pd.DataFrame) -> pd.DataFrame:
    """Standardise column names and fix the specific data quality issues
    identified during profiling. Nothing is silently dropped — problem
    rows are flagged, not deleted."""
    df = raw.rename(columns=RENAME_MAP)
    # any leftover raw column names (Year, Week No., etc.) get snake_cased
    df.columns = [c if c in RENAME_MAP.values() else _snake(c) for c in df.columns]

    # Issue 0: fully identical duplicate rows
    n_before = len(df)
    df = df.drop_duplicates()
    print(f"Removed {n_before - len(df)} fully duplicate row(s).")

    # Issue 1: VA%/Mup% hold "#DIV/0!" text where Sell Price = 0
    df["va_pct"] = pd.to_numeric(df["va_pct"], errors="coerce")
    df["mup_pct"] = pd.to_numeric(df["mup_pct"], errors="coerce")
    df["zero_price_job_flag"] = df["sell_price"] == 0

    # Issue 2: blank Binding Type is a valid category per the field
    df["binding_type"] = df["binding_type"].fillna("Outsourced/Not applicable")

    # Issue 3 & 6: convert to plain dates first
    df["sales_in"] = pd.to_datetime(df["sales_in"]).dt.normalize()
    df["sales_out"] = pd.to_datetime(df["sales_out"]).dt.normalize()
    df["ship_date"] = pd.to_datetime(df["ship_date"]).dt.normalize()
    df["date_anomaly_flag"] = df["ship_date"].notna() & df["sales_in"].notna() & (df["ship_date"] < df["sales_in"])

    # Issue 4: negative VA amount / sell price = credits, reworks, goodwill jobs
    df["credit_or_rework_flag"] = (df["va_amount"] < 0) | (df["sell_price"] < 0)

    # Issue 5: Product Type
    df["product_type_clean"] = df["product_type"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    lower = df["product_type_clean"].str.lower()
    df.loc[lower.str.contains("brochures", na=False), "product_type_clean"] = "Brochures / Price List"
    df.loc[lower.str.contains("leaflets to a4", na=False), "product_type_clean"] = "Leaflets to A4 / Price Lists"

    return df


# 2.5 DATA VALIDATION ASSERTIONS
def run_data_validation_checks(df: pd.DataFrame) -> None:
    """Systematic business-rule checks: state what's logically impossible
    for each field, then count violations. Doesn't alter the data."""
    print("---- Data validation assertions ----")
    print("Ship date before SalesIn (impossible):     ", df["date_anomaly_flag"].sum())
    print("Negative Quantity:                          ", (df["quantity"] < 0).sum())
    print("Negative Sell Price:                        ", (df["sell_price"] < 0).sum())
    print("Zero-price jobs (VA%/Mup% undefined):       ", df["zero_price_job_flag"].sum())
    print("Negative VA Amount / credit-or-rework rows: ", df["credit_or_rework_flag"].sum())
    print("SalesOut before SalesIn (impossible):       ", (df["sales_out"] < df["sales_in"]).sum())


# 3. JOB-LEVEL REORDER HISTORY
def build_job_history(df: pd.DataFrame) -> pd.DataFrame:
    """`title` is not a unique job ID — it's a recurring job/product code.
    Builds the gap-between-orders table the reorder prediction is based on."""
    jh = df[df["sales_in"].notna()].sort_values(["title", "sales_in"]).copy()
    jh["gap_days"] = jh.groupby("title")["sales_in"].diff().dt.days
    return jh


def summarise_reorder_windows(job_history: pd.DataFrame, min_orders: int = MIN_ORDERS_FOR_REORDER_WINDOW) -> pd.DataFrame:
    """Per-Title reorder statistics, restricted to Titles with enough
    history to trust a cadence estimate (>= 3 orders = 2+ gaps)."""
    grp = job_history.groupby(["title", "customer_id", "customer_name"])
    out = grp.agg(
        n_orders=("sales_in", "size"),
        mean_gap=("gap_days", "mean"),
        sd_gap=("gap_days", "std"),
        last_order=("sales_in", "max"),
    ).reset_index()
    out = out[out["n_orders"] >= min_orders].copy()
    out["predicted_next_order"] = out["last_order"] + pd.to_timedelta(out["mean_gap"].round(), unit="D")
    return out


# 4. CUSTOMER-LEVEL CADENCE & CHURN
def summarise_customer_cadence(df: pd.DataFrame, ref_date=None, k: float = CHURN_K) -> pd.DataFrame:
    """Order cadence per customer across ALL their orders (not just repeat
    Titles), so low-frequency customers still get a churn read."""
    if ref_date is None:
        ref_date = df["sales_in"].max()

    d = df[df["sales_in"].notna()].sort_values(["customer_id", "sales_in"]).copy()
    d["gap_days"] = d.groupby("customer_id")["sales_in"].diff().dt.days

    out = d.groupby("customer_id").agg(
        customer_name=("customer_name", "first"),
        n_orders=("sales_in", "size"),
        mean_gap=("gap_days", "mean"),
        sd_gap=("gap_days", "std"),
        first_order=("sales_in", "min"),
        last_order=("sales_in", "max"),
    ).reset_index()

    # Customers with only 1-2 orders won't have a reliable SD
    out["sd_gap_filled"] = out["sd_gap"].fillna(out["mean_gap"] * 0.5)
    out["days_since_last"] = (ref_date - out["last_order"]).dt.days

    # Sensitivity check: report the flag at both k=1 and a stricter k=1.5
    out["churn_threshold"] = out["mean_gap"] + (k * out["sd_gap_filled"])
    out["at_risk"] = out["days_since_last"] > out["churn_threshold"]
    out["churn_threshold_1_5sd"] = out["mean_gap"] + (1.5 * out["sd_gap_filled"])
    out["at_risk_1_5sd"] = out["days_since_last"] > out["churn_threshold_1_5sd"]
    return out


def summarise_customer_value(df: pd.DataFrame, cadence: pd.DataFrame, eur_to_gbp: float = EUR_TO_GBP) -> pd.DataFrame:
    """Customer value (total VA, VA share, job count), joined onto cadence.

    IMPORTANT: every customer trades in a single currency, but customers
    differ from EACH OTHER in currency. Summing/ranking raw sell_price /
    va_amount across customers would silently blend Sterling and Euro as if
    equal. Keeps the native-currency total AND adds total_va_gbp_equiv,
    which is the only column that should be used for cross-customer
    ranking, sums, or a single headline number.
    """
    value = df.groupby(["customer_id", "currency"]).agg(
        total_va=("va_amount", "sum"),
        total_sell=("sell_price", "sum"),
        n_jobs=("va_amount", "size"),
    ).reset_index()

    value["total_va_gbp_equiv"] = np.where(
        value["currency"] == "Euro", value["total_va"] * eur_to_gbp, value["total_va"]
    )
    value["va_pct_of_total"] = value["total_va_gbp_equiv"] / value["total_va_gbp_equiv"].sum() * 100

    merged = cadence.merge(value, on="customer_id", how="left")
    return merged.sort_values(["at_risk", "total_va_gbp_equiv"], ascending=[False, False])


# 5. SUPPORTING CUTS
def summarise_work_type_margin(df: pd.DataFrame) -> pd.DataFrame:
    """VA% by Work Type — feeds the 'where should we invest' recommendation."""
    d = df[df["sell_price"] > 0]  # excludes the #DIV/0! rows cleanly
    out = d.groupby("work_type").agg(
        n_jobs=("va_amount", "size"),
        total_va=("va_amount", "sum"),
        mean_va_pct=("va_pct", "mean"),
    ).reset_index()
    return out.sort_values("total_va", ascending=False)


def summarise_product_type_margin(df: pd.DataFrame) -> pd.DataFrame:
    """VA% by cleaned Product Type — finer-grained than Work Type."""
    d = df[df["sell_price"] > 0]
    out = d.groupby("product_type_clean").agg(
        n_jobs=("va_amount", "size"),
        total_va=("va_amount", "sum"),
        mean_va_pct=("va_pct", "mean"),
    ).reset_index()
    return out.sort_values("total_va", ascending=False)


def summarise_customer_concentration(customer_value: pd.DataFrame) -> pd.DataFrame:
    """How many customers drive 80% of VA. Uses the GBP-equivalent total
    so the ranking isn't distorted by currency mixing."""
    out = customer_value.sort_values("total_va_gbp_equiv", ascending=False).copy()
    out["cume_va_gbp"] = out["total_va_gbp_equiv"].cumsum()
    out["cume_pct"] = out["cume_va_gbp"] / out["total_va_gbp_equiv"].sum() * 100
    out["customer_rank"] = range(1, len(out) + 1)
    return out[["customer_rank", "customer_id", "customer_name", "total_va_gbp_equiv", "cume_pct"]]


def summarise_delivery_time(df: pd.DataFrame) -> pd.DataFrame:
    """Delivery time (SalesIn -> Ship date), by Work Type. Excludes rows
    flagged as date anomalies so they don't distort the median/mean."""
    d = df[df["sales_in"].notna() & df["ship_date"].notna() & ~df["date_anomaly_flag"]].copy()
    d["days_to_ship"] = (d["ship_date"] - d["sales_in"]).dt.days
    out = d.groupby("work_type").agg(
        n_jobs=("days_to_ship", "size"),
        median_days_to_ship=("days_to_ship", "median"),
        mean_days_to_ship=("days_to_ship", "mean"),
    ).reset_index()
    return out.sort_values("median_days_to_ship", ascending=False)


def summarise_new_vs_retained(df: pd.DataFrame) -> pd.DataFrame:
    """New vs retained CUSTOMERS by year (counts distinct customers, not
    order rows — a customer with 40 orders in a year counts once).

    CAVEAT: 2023 is the first year in this extract, so every customer active
    that year is labelled 'New' by construction — treat that figure as a
    data-window artefact, not genuine acquisition, and lead with 2024
    onwards for real growth claims.
    """
    first_year = df.groupby("customer_id")["sales_in"].min().dt.year.rename("first_year")
    d = df.merge(first_year, on="customer_id", how="left")
    d["order_year"] = d["sales_in"].dt.year
    d["customer_status"] = np.where(d["order_year"] == d["first_year"], "New", "Retained")
    d = d[["customer_id", "order_year", "customer_status"]].drop_duplicates()
    return d.groupby(["order_year", "customer_status"]).size().reset_index(name="n_customers")


# 5.5 SIX-MONTH REVENUE OUTLOOK
def summarise_revenue_forecast(df: pd.DataFrame, months_ahead: int = 6) -> pd.DataFrame:
    """Monthly revenue (sum of sell_price), with a simple trend + seasonal
    projection for the next `months_ahead` months.

    Deliberately the same 'descriptive statistics, not machine learning'
    approach as the rest of the pipeline: a linear trend fitted across all
    months, plus a multiplicative seasonal index per calendar month (how
    much higher or lower that month typically runs vs the trend line).
    No black-box model, and the whole thing is checkable by hand.

    Honesty check built in: the last `months_ahead` months of ACTUAL history
    are held out, the model is fit on everything before that, and the held-
    out months are predicted and compared to what really happened. That
    backtest error (mean absolute percentage error) is then used to draw an
    honest error band around the forward-looking forecast, rather than
    presenting a single confident number.
    """
    monthly = (
        df[df["sales_in"].notna()]
        .assign(month=lambda x: x["sales_in"].dt.to_period("M"))
        .groupby("month")["sell_price"].sum()
        .sort_index()
    )

    def fit_trend_seasonal(series: pd.Series):
        t = np.arange(len(series))
        slope, intercept = np.polyfit(t, series.values, 1)
        trend = intercept + slope * t
        seasonal_ratio = series.values / trend
        seasonal = pd.Series(seasonal_ratio, index=series.index.month).groupby(level=0).mean()
        seasonal = seasonal / seasonal.mean()  # normalise so the average factor is 1
        return intercept, slope, seasonal

    # ---- Backtest: hold out the most recent `months_ahead` actual months ----
    if len(monthly) <= months_ahead + 6:
        raise ValueError("Not enough monthly history to backtest a 6-month forecast reliably.")

    train = monthly.iloc[:-months_ahead]
    test = monthly.iloc[-months_ahead:]
    intercept, slope, seasonal = fit_trend_seasonal(train)

    t_test = np.arange(len(train), len(train) + months_ahead)
    pred_test = (intercept + slope * t_test) * seasonal.reindex(test.index.month).values
    backtest_mape = float(np.mean(np.abs((test.values - pred_test) / test.values)) * 100)

    # ---- Final model: fit on ALL available months ----
    intercept_f, slope_f, seasonal_f = fit_trend_seasonal(monthly)
    t_all = np.arange(len(monthly))
    model_fit = (intercept_f + slope_f * t_all) * seasonal_f.reindex(monthly.index.month).values

    future_idx = pd.period_range(monthly.index[-1] + 1, periods=months_ahead, freq="M")
    t_future = np.arange(len(monthly), len(monthly) + months_ahead)
    forecast = (intercept_f + slope_f * t_future) * seasonal_f.reindex(future_idx.month).values

    history = pd.DataFrame({
        "month": monthly.index.astype(str),
        "actual": monthly.values,
        "model_fit": model_fit,
        "forecast": np.nan,
        "forecast_low": np.nan,
        "forecast_high": np.nan,
        "is_forecast": False,
    })
    future = pd.DataFrame({
        "month": future_idx.astype(str),
        "actual": np.nan,
        "model_fit": np.nan,
        "forecast": forecast,
        "forecast_low": forecast * (1 - backtest_mape / 100),
        "forecast_high": forecast * (1 + backtest_mape / 100),
        "is_forecast": True,
    })
    out = pd.concat([history, future], ignore_index=True)
    out["backtest_mape"] = backtest_mape  # same value every row, simplest way to keep it in the CSV
    return out


# 6. CHART EXPORTS
def generate_charts(customer_value: pd.DataFrame, work_type_va: pd.DataFrame,
                     concentration: pd.DataFrame, revenue_forecast: pd.DataFrame,
                     chart_dir: Path = CHART_DIR) -> None:
    """Same visualisations shown in the board deck, generated straight
    from the pipeline output. Re-running the pipeline regenerates all of
    them, so charts dropped into the README never drift out of sync with
    the data."""
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates

    chart_dir.mkdir(parents=True, exist_ok=True)
    INK, ACCENT, GREY = "#1B2A41", "#7A1F2B", "#D8DCE3"
    plt.rcParams["font.size"] = 11

    # 1. Concentration curve
    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.plot(concentration["customer_rank"], concentration["cume_pct"], color=INK, linewidth=2.2)
    ax.axhline(80, linestyle="--", color=GREY)
    ax.set_ylim(0, 100)
    ax.set_xlabel("Customer rank")
    ax.set_ylabel("Cumulative % of VA")
    ax.set_title("Cumulative share of total value-added, customers ranked highest first", fontsize=10.5, color="#3B4453")
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(chart_dir / "customer_concentration.png", dpi=200, facecolor="white")
    plt.close(fig)

    # 2. At-risk value, doughnut
    at_risk_va = customer_value.loc[customer_value["at_risk"], "total_va_gbp_equiv"].sum()
    rest_va = customer_value.loc[~customer_value["at_risk"], "total_va_gbp_equiv"].sum()
    fig, ax = plt.subplots(figsize=(5, 5))
    wedges, _, autotexts = ax.pie(
        [at_risk_va, rest_va], colors=[ACCENT, GREY], startangle=90,
        autopct="%.1f%%", pctdistance=0.8,
        wedgeprops=dict(width=0.4, edgecolor="white"),
    )
    for t in autotexts:
        t.set_color("white")
        t.set_fontweight("bold")
    ax.set_title("Value added, GBP-equivalent", fontsize=10.5, color="#3B4453")
    ax.legend(wedges, ["At-risk (k=1)", "Rest of book"], loc="lower center",
              bbox_to_anchor=(0.5, -0.08), ncol=2, frameon=False)
    fig.tight_layout()
    fig.savefig(chart_dir / "at_risk_share.png", dpi=200, facecolor="white")
    plt.close(fig)

    # 3. Mean VA% by work type
    wt = work_type_va.sort_values("mean_va_pct", ascending=False)
    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    bars = ax.bar(wt["work_type"], wt["mean_va_pct"] * 100, color=INK, width=0.6)
    ax.bar_label(bars, fmt="%.0f%%", padding=3)
    ax.set_ylabel("Mean VA%")
    ax.set_ylim(0, wt["mean_va_pct"].max() * 100 * 1.15)
    ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    fig.savefig(chart_dir / "margin_by_work_type.png", dpi=200, facecolor="white")
    plt.close(fig)

    # 4. Six-month revenue outlook: actual, model fit, forecast + error band
    rf = revenue_forecast.copy()
    rf["month_dt"] = pd.PeriodIndex(rf["month"], freq="M").to_timestamp()
    mape = rf["backtest_mape"].iloc[0]
    forecast_total = rf.loc[rf["is_forecast"], "forecast"].sum()

    fig, ax = plt.subplots(figsize=(9, 4.8))
    ax.plot(rf["month_dt"], rf["actual"], color=INK, linewidth=2, label="Actual")
    ax.plot(rf["month_dt"], rf["model_fit"], color="#8A93A3", linewidth=1.2, linestyle=":", label="Model fit")

    # connect the last actual point to the first forecast point so the line doesn't gap
    fc = rf[rf["is_forecast"]].copy()
    last_actual_row = rf[~rf["is_forecast"]].iloc[[-1]]
    fc_plot = pd.concat([
        last_actual_row.assign(forecast=last_actual_row["actual"], forecast_low=last_actual_row["actual"], forecast_high=last_actual_row["actual"]),
        fc,
    ])
    ax.plot(fc_plot["month_dt"], fc_plot["forecast"], color=ACCENT, linewidth=2.4, label="Forecast")
    ax.fill_between(fc_plot["month_dt"], fc_plot["forecast_low"], fc_plot["forecast_high"],
                     color=ACCENT, alpha=0.15, label=f"Backtest error band (\u00b1{mape:.0f}%)")

    ax.set_ylabel("Revenue (\u00a3 per month)")
    ax.set_title(
        f"Six-month revenue outlook: next six months around \u00a3{forecast_total/1e6:.2f}M "
        f"(backtest MAPE {mape:.0f}%)",
        fontsize=11, color="#3B4453",
    )
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper left", frameon=False, fontsize=9)
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(chart_dir / "revenue_forecast.png", dpi=200, facecolor="white")
    plt.close(fig)

    print(
        f"Charts written to {chart_dir}: customer_concentration.png, at_risk_share.png, "
        f"margin_by_work_type.png, revenue_forecast.png"
    )


# RUN PIPELINE
def run_pipeline(input_dir: Path = INPUT_DIR, output_dir: Path = OUTPUT_DIR) -> dict:
    raw = load_raw_data(input_dir)

    print("\n============ RAW DATA PROFILE ============")
    profile_data(raw)

    df = clean_data(raw)

    print("\n============ POST-CLEAN VALIDATION ============")
    run_data_validation_checks(df)

    print(f"Rows loaded: {len(df)}")
    print(f"Div/0 rows (va_pct now NA): {(df['va_pct'].isna() & (df['sell_price'] == 0)).sum()}")
    print(f"Date anomalies flagged: {df['date_anomaly_flag'].sum()}")
    print(f"Credit/rework rows flagged: {df['credit_or_rework_flag'].sum()}")

    job_history = build_job_history(df)
    reorder_windows = summarise_reorder_windows(job_history)
    cadence = summarise_customer_cadence(df)
    customer_value = summarise_customer_value(df, cadence)
    concentration = summarise_customer_concentration(customer_value)
    delivery_time = summarise_delivery_time(df)
    work_type_va = summarise_work_type_margin(df)
    product_type_va = summarise_product_type_margin(df)
    new_vs_retained = summarise_new_vs_retained(df)
    revenue_forecast = summarise_revenue_forecast(df)

    covered_customers = reorder_windows["customer_id"].nunique()
    total_customers = df["customer_id"].nunique()

    output_dir.mkdir(parents=True, exist_ok=True)
    reorder_windows.to_csv(output_dir / "reorder_windows.csv", index=False)
    customer_value.to_csv(output_dir / "customer_value_and_churn.csv", index=False)
    concentration.to_csv(output_dir / "customer_concentration.csv", index=False)
    delivery_time.to_csv(output_dir / "delivery_time_by_work_type.csv", index=False)
    work_type_va.to_csv(output_dir / "work_type_margin.csv", index=False)
    product_type_va.to_csv(output_dir / "product_type_margin.csv", index=False)
    new_vs_retained.to_csv(output_dir / "new_vs_retained.csv", index=False)
    revenue_forecast.to_csv(output_dir / "revenue_forecast_6m.csv", index=False)

    generate_charts(customer_value, work_type_va, concentration, revenue_forecast)

    at_risk_value = customer_value.loc[customer_value["at_risk"], "total_va_gbp_equiv"].sum()
    total_value = customer_value["total_va_gbp_equiv"].sum()
    print(
        f"At-risk customers (k=1): {customer_value['at_risk'].sum()} of {len(customer_value)} | "
        f"VA at risk: GBP-equiv {at_risk_value:,.0f} ({at_risk_value / total_value * 100:.1f}% of total)"
    )
    print(f"At-risk customers (stricter k=1.5): {customer_value['at_risk_1_5sd'].sum()} of {len(customer_value)}")
    print(
        f"Job-level reorder window coverage: {covered_customers} of {total_customers} customers "
        f"({covered_customers / total_customers * 100:.0f}%) have a Title with >={MIN_ORDERS_FOR_REORDER_WINDOW} orders"
    )
    forecast_total = revenue_forecast.loc[revenue_forecast["is_forecast"], "forecast"].sum()
    forecast_mape = revenue_forecast["backtest_mape"].iloc[0]
    print(
        f"Six-month revenue outlook: \u00a3{forecast_total:,.0f} total (backtest MAPE {forecast_mape:.1f}%)"
    )

    return dict(
        df=df, reorder_windows=reorder_windows, customer_value=customer_value,
        concentration=concentration, delivery_time=delivery_time,
        work_type_va=work_type_va, product_type_va=product_type_va,
        new_vs_retained=new_vs_retained, revenue_forecast=revenue_forecast,
    )


if __name__ == "__main__":
    # Place the source .xlsx in data/input/ before running this.
    results = run_pipeline()




============ RAW DATA PROFILE ============
---- Null counts (top 10) ----
Binding Type    2642
VA%              218
Ship date        216
SalesOut         129
manadj            64
mupnett           64
Mup%              14
Handling          12
Puchases          12
Product Type       1
dtype: int64

---- Fully identical duplicate rows ----
1 extra copies ( 2 rows total involved )

---- Repeated Title values (legitimate reorders, not errors) ----
1766 rows across 651 distinct Titles

---- #DIV/0! rows ----
VA%:  0
Mup%: 0

---- Ship date before SalesIn (impossible) ----
9 rows
Removed 1 fully duplicate row(s).

============ POST-CLEAN VALIDATION ============
---- Data validation assertions ----
Ship date before SalesIn (impossible):      9
Negative Quantity:                           0
Negative Sell Price:                         6
Zero-price jobs (VA%/Mup% undefined):        218
Negative VA Amount / credit-or-rework rows:  277
SalesOut before SalesIn (impossible):        0
Rows loaded: 6